In [1]:
import pandas as pd

df = pd.read_csv("cpi_monthly.csv", dtype={"series_code": str})
df

,period,series_code,index_value
0,2018-01-01,0001,88.9
1,2018-01-01,0161,88.9
2,2018-01-01,0178,89.7
3,2018-02-01,0001,88.9
4,2018-02-01,0161,89.1
...,...,...,...
310,2026-08-01,0161,102.0
311,2026-08-01,0178,102.5
312,2026-09-01,0001,102.2
313,2026-09-01,0161,102.3


In [3]:
import plotly.express as px

df["period"] = pd.to_datetime(df["period"])

df["series"] = df["series_code"].map({
    "0001": "CPI",
    "0161": "CCPI",
    "0178": "CCCPI"
})

fig = px.line(
    df.sort_values("period"),
    x="period",
    y="index_value",
    color="series",
    title="Japan CPI Trends (2018-2026)",
    labels={
        "period": "Year",
        "index_value": "Index (2025=100)",
        "series": "Indicator"
    }
)

fig.show()

In [5]:
df = df.sort_values(["series", "period"])

df["yoy"] = (
    df.groupby("series")["index_value"]
    .pct_change(12) * 100
)

fig = px.line(
    df,
    x="period",
    y="yoy",
    color="series",
    title="Japan CPI - Year-on-Year Change",
    labels={
        "period": "Year",
        "yoy": "Change (%)",
        "series": "Indicator"
    }
)

fig.add_hline(y=0, line_dash="dash")
fig.show()

In [9]:
gap = df.pivot(
    index="period",
    columns="series",
    values="yoy"
)

gap["Fresh Food"] = gap["CPI"] - gap["CCPI"]
gap["Energy"] = gap["CCPI"] - gap["CCCPI"]

fig = px.line(
    gap,
    y=["Fresh Food", "Energy"],
    title="Inflation Rate Gaps",
    labels={
        "period": "Year",
        "value": "Gap (pp)",
        "variable": "Category"
    }
)

fig.add_hline(y=0, line_dash="dash")
fig.show()

In [11]:
df["year"] = df["period"].dt.year

annual = (
    df.groupby(["year", "series"])["yoy"]
    .mean()
    .reset_index()
    .dropna()
)

fig = px.bar(
    annual,
    x="year",
    y="yoy",
    color="series",
    barmode="group",
    title="Annual Average Inflation Rate",
    labels={
        "year": "Year",
        "yoy": "Inflation Rate (%)",
        "series": "Indicator"
    }
)

fig.show()

<a style='text-decoration:none;line-height:16px;display:flex;color:#5B5B62;padding:10px;justify-content:end;' href='https://deepnote.com?utm_source=created-in-deepnote-cell&projectId=8927433d-090f-4dbd-9395-a114398bf760' target="_blank">

Created in <span style='font-weight:600;margin-left:4px;'>Deepnote</span></a>